# CMFD Convergence and Diagnostics

This tutorial covers when a CMFD solve stops, how CMFD solves its coarse problem, and how to read
its diagnostics:

* the CMFD balance check and `balance_residual_tolerance`,
* the coarse eigenvalue iteration (`pi_max_its`, `pi_k_tol` and `pi_phi_tol`),
* the coarse linear solver (`coarse_solver_policy`),
* `verbose=True` diagnostics and the solver accessors,
* skipped corrections and `inactive_iterations`,
* CMFD for the adjoint problem.

## The C5G7 model

The C5G7 benchmark [Lewis et al., NEA/NSC/DOC(2003)16] is a quarter core of four 17 x 17 pin
assemblies and a water reflector. The UO$_2$ assemblies sit on the diagonal. The MOX assemblies
have three enrichment zones (4.3%, 7.0% and 8.7%). Every assembly has 24 guide tubes and a central
fission chamber. The pin pitch is 1.26 cm and the reflector is one assembly (21.42 cm) thick. The
core center is in the lower-left corner, so the left and bottom boundaries are reflecting and the
right and top boundaries are vacuum. The benchmark provides seven-group cross sections with
upscattering in the four thermal groups.

The tutorials in this series use two models of that core.

* The **pin-homogenized model** mixes each fuel pin with its moderator by volume, so every pin
  cell is one 1.26 cm square mesh cell. It keeps the assembly layout, the MOX zoning, the guide
  tubes and the reflector, which are what make C5G7 hard for unaccelerated power iteration, and
  it solves in a few seconds. The option studies use it.
* The **heterogeneous model** in the [benchmark tutorial](c5g7_benchmark.ipynb) represents each
  pin by a square of the same area as the round pin, surrounded by moderator.

Both use a 2 x 2 KBA partition, so run the generated scripts with four MPI processes. A light
quadrature keeps the run times short. Production calculations need more directions.

## Model and measurement code

The next two cells build the C5G7 problem and measure a solve. They are the same in every tutorial
of this series. `solve()` builds a fresh problem for each case, so every solve starts from the
same state. It returns the eigenvalue, the number of power iterations (outers), the number of
transport sweeps, the number of CMFD coarse cells, the number of skipped CMFD corrections, whether
the solve converged, the wall time, and the fission rate of every fuel pin normalized to the
average fuel pin. The wall time covers building the problem, initializing the solver, and the
solve. Building the problem is the same in every case.

In [ ]:
import math
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.fieldfunc import FieldFunctionGridBased
from pyopensn.math import Vector3
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.solver import (
    CMFDAcceleration,
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
xs_dir = next(
    path / "test/assets/xs/c5g7"
    for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs/c5g7").is_dir()
)

In [ ]:
PITCH = 1.26  # pin pitch (cm)
PIN_RADIUS = 0.54  # fuel pin radius (cm)
ASSEMBLY_PINS = 17
CORE_PINS = 2 * ASSEMBLY_PINS  # fuel-assembly pins per side of the quarter core
DOMAIN_PINS = 3 * ASSEMBLY_PINS  # including the water reflector

WATER, UO2, MOX_43, MOX_70, MOX_87, GUIDE_TUBE, FISSION_CHAMBER = range(7)
XS_FILES = [
    "XS_water",
    "XS_UO2",
    "XS_4_3pMOX",
    "XS_7pMOX",
    "XS_8_7pMOX",
    "XS_guide_tube",
    "XS_fission_chamber",
]
FUEL = (UO2, MOX_43, MOX_70, MOX_87)
GUIDE_TUBES = {
    (2, 5), (2, 8), (2, 11), (3, 3), (3, 13), (5, 2), (5, 5), (5, 8), (5, 11), (5, 14),
    (8, 2), (8, 5), (8, 11), (8, 14), (11, 2), (11, 5), (11, 8), (11, 11), (11, 14),
    (13, 3), (13, 13), (14, 5), (14, 8), (14, 11),
}
# One quarter of the MOX assembly, mirrored about its center row and column.
# 1, 2 and 3 are 4.3%, 7.0% and 8.7% MOX, G is a guide tube and F the fission chamber.
MOX_QUARTER = [
    "111111111",
    "122222222",
    "12222G22G",
    "122G23333",
    "122233333",
    "12G33G33G",
    "122333333",
    "122333333",
    "12G33G33F",
]
MOX_ZONES = {"1": MOX_43, "2": MOX_70, "3": MOX_87}


def pin_material(px, py):
    """Return the material of the pin cell in column px and row py of the domain."""
    if px >= CORE_PINS or py >= CORE_PINS:
        return WATER
    i, j = py % ASSEMBLY_PINS, px % ASSEMBLY_PINS
    if (i, j) == (8, 8):
        return FISSION_CHAMBER
    if (i, j) in GUIDE_TUBES:
        return GUIDE_TUBE
    if px // ASSEMBLY_PINS == py // ASSEMBLY_PINS:
        return UO2
    return MOX_ZONES[MOX_QUARTER[min(i, 16 - i)][min(j, 16 - j)]]


def load_cross_sections(homogenized):
    """Load the seven C5G7 materials, mixed with water over the pin cell if homogenized."""
    cross_sections = []
    for name in XS_FILES:
        xs = MultiGroupXS()
        xs.LoadFromOpenSn(str(xs_dir / f"{name}.xs"))
        cross_sections.append(xs)
    if not homogenized:
        return cross_sections
    fuel_fraction = math.pi * PIN_RADIUS**2 / PITCH**2
    water = cross_sections[WATER]
    return [water] + [
        MultiGroupXS.Combine([(xs, fuel_fraction), (water, 1.0 - fuel_fraction)])
        for xs in cross_sections[1:]
    ]


def make_mesh(homogenized):
    """Build the quarter core on a 2 x 2 KBA partition, so run with four MPI processes."""
    if homogenized:
        nodes = [PITCH * p for p in range(DOMAIN_PINS + 1)]
    else:
        side = math.sqrt(math.pi) * PIN_RADIUS  # square pin with the area of the round one
        gap = 0.5 * (PITCH - side)
        nodes = [PITCH * p + offset for p in range(CORE_PINS) for offset in (0.0, gap, gap + side)]
        nodes += [PITCH * (CORE_PINS + 0.5 * k) for k in range(2 * ASSEMBLY_PINS + 1)]
    cut = PITCH * ASSEMBLY_PINS
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes],
        partitioner=KBAGraphPartitioner(nx=2, ny=2, xcuts=[cut], ycuts=[cut]),
    ).Execute()
    mesh.SetOrthogonalBoundaries()

    def block_id(centroid, old_block_id):
        px, py = int(centroid.x // PITCH), int(centroid.y // PITCH)
        if not homogenized:
            local_x, local_y = centroid.x - px * PITCH, centroid.y - py * PITCH
            if not (gap < local_x < gap + side and gap < local_y < gap + side):
                return WATER
        return pin_material(px, py)

    mesh.SetBlockIDFromFunction(block_id)
    return mesh


def make_problem(
    homogenized=True,
    n_azimuthal=4,
    group_ranges=((0, 6),),
    inner_linear_method="classic_richardson",
    l_max_its=4,
    adjoint=False,
    verbose=False,
):
    """Build the C5G7 quarter-core transport problem."""
    cross_sections = load_cross_sections(homogenized)
    quadrature = GLCProductQuadrature2DXY(
        n_polar=2, n_azimuthal=n_azimuthal, scattering_order=0
    )
    problem = DiscreteOrdinatesProblem(
        mesh=make_mesh(homogenized),
        num_groups=7,
        groupsets=[
            {
                "groups_from_to": group_range,
                "angular_quadrature": quadrature,
                "inner_linear_method": inner_linear_method,
                "l_max_its": l_max_its,
                "l_abs_tol": 1.0e-10,
            }
            for group_range in group_ranges
        ],
        xs_map=[{"block_ids": [m], "xs": xs} for m, xs in enumerate(cross_sections)],
        boundary_conditions=[
            {"name": "xmin", "type": "reflecting"},
            {"name": "ymin", "type": "reflecting"},
        ],
        options={
            "adjoint": adjoint,
            "use_precursors": False,
            "power_default_kappa": 1.0,
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": verbose,
        },
    )
    return problem, cross_sections

In [ ]:
def pin_fission_rates(problem, cross_sections):
    """Return the fission rate of every fuel pin, normalized to the average fuel pin.

    Each pin center is the center of a rectangular cell, where the bilinear PWLD flux equals
    its cell average.
    """
    pins = [
        (px, py)
        for py in range(CORE_PINS)
        for px in range(CORE_PINS)
        if pin_material(px, py) in FUEL
    ]
    points = [Vector3((px + 0.5) * PITCH, (py + 0.5) * PITCH, 0.0) for px, py in pins]
    flux = FieldFunctionGridBased.GetPointValues(problem.GetScalarFluxFieldFunction(), points)
    rates = []
    for n, (px, py) in enumerate(pins):
        sigma_f = cross_sections[pin_material(px, py)].sigma_f
        rates.append(sum(sigma_f[g] * flux[g][n][0] for g in range(len(sigma_f))))
    mean = sum(rates) / len(rates)
    return {pin: rate / mean for pin, rate in zip(pins, rates)}


def solve(label, cmfd=None, k_tol=1.0e-7, max_iters=400, fission_source_tol=None, **problem_options):
    """Build a fresh problem, solve it with power iteration, and return the metrics.

    cmfd holds the CMFDAcceleration options. Without it the power iteration is unaccelerated.
    """
    comm.Barrier()
    start = MPI.Wtime()
    problem, cross_sections = make_problem(**problem_options)
    acceleration = None
    solver_options = {"problem": problem, "k_tol": k_tol, "max_iters": max_iters}
    if fission_source_tol is not None:
        solver_options["fission_source_tol"] = fission_source_tol
    if cmfd is not None:
        acceleration = CMFDAcceleration(problem=problem, **cmfd)
        solver_options["acceleration"] = acceleration
    solver = PowerIterationKEigenSolver(**solver_options)
    solver.Initialize()
    solver.Execute()
    seconds = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return {
        "label": label,
        "k": solver.GetEigenvalue(),
        "outers": solver.GetNumPowerIterations(),
        "sweeps": solver.GetNumSweeps(),
        "converged": solver.IsConverged(),
        "coarse_cells": acceleration.GetNumCoarseCells() if acceleration else 0,
        "skipped": acceleration.GetNumSkippedCorrections() if acceleration else 0,
        "seconds": seconds,
        "pins": pin_fission_rates(problem, cross_sections),
    }


def report(results, k_reference, pin_reference=None):
    """Print one row per solve and one stable key per metric for the regression tests."""
    if rank != 0:
        return
    header = f"{'case':<34}{'k_eff':>12}{'k error':>10}{'outers':>8}{'sweeps':>8}"
    header += f"{'coarse':>8}{'skips':>6}{'conv':>5}"
    header += f"{'pin error':>10}" if pin_reference else ""
    print(header + f"{'time (s)':>10}")
    for r in results:
        k_error = r["k"] - k_reference
        row = f"{r['label']:<34}{r['k']:>12.8f}{k_error:>10.1e}{r['outers']:>8}{r['sweeps']:>8}"
        row += f"{r['coarse_cells']:>8}{r['skipped']:>6}{int(r['converged']):>5}"
        if pin_reference:
            row += f"{max_pin_error(r, pin_reference):>10.1e}"
        print(row + f"{r['seconds']:>10.2f}")
    for r in results:
        key = r["label"].replace(" ", "_")
        print(f"{key} k-effective error={r['k'] - k_reference:.6e}")
        print(f"{key} converged={int(r['converged'])}")


def max_pin_error(result, pin_reference):
    """Largest difference of the normalized pin fission rates from the reference."""
    return max(abs(result["pins"][pin] - value) for pin, value in pin_reference.items())

## A tightly converged reference

The reference is a CMFD solve with the series settings and tolerances far tighter than the ones
used in the cases below. Its eigenvalue and pin fission rates measure the error of each case.

In [ ]:
TUNED = {
    "aggregation_size": 4,
    "allow_mixed_material_aggregation": True,
    "group_aggregation_size": 4,
    "update_wgs_max_its": 4,
}

reference = solve(
    "reference",
    cmfd={**TUNED, "balance_residual_tolerance": 1.0e-10},
    k_tol=1.0e-11,
)
K_REFERENCE = reference["k"]
PIN_REFERENCE = reference["pins"]

## The balance check

Power iteration with CMFD stops only when four conditions hold. The relative eigenvalue change
must pass `k_tol (1 - rho)`, with `rho` the convergence rate estimated from the change of the
flux, in two consecutive power iterations. The root-mean-square relative change of the cell
fission sources must pass `fission_source_tol (1 - rho)`. The CMFD balance check must be
satisfied **and** the CMFD correction of that power iteration must have been applied, not
skipped. The balance check restricts
the transport solution to the coarse mesh and measures how well it satisfies the coarse balance
equations with the transport currents. It is evaluated before the correction, from the restricted
flux and currents of the latest sweep. Its fission term applies the production cross sections,
homogenized with the latest sweep's flux, to the restricted flux before that sweep. Its measure is
the relative L2 norm of the coarse balance residual. The
eigenvalue converges much faster than the flux shape, so stopping on the eigenvalue alone can leave
the flux, and the pin powers, less converged than the eigenvalue suggests.

The residual is a consistency check of the CMFD problem, not an estimate of the eigenvalue error.
The default tolerance, `1e-6`, is a good starting point. An unnecessarily tight value can force
iterations past the point where the requested accuracy has been reached, especially on large 3D
problems.

The fission-source test also guards the flux shape, and on this problem it gives the accuracy of
the default balance tolerance by itself. The first two solves below therefore turn it off
(`fission_source_tol=0`) to show the balance check alone. The third uses the defaults.

In [ ]:
balance_results = [
    solve(
        "balance 1e-2",
        cmfd={**TUNED, "balance_residual_tolerance": 1.0e-2},
        fission_source_tol=0.0,
    ),
    solve("balance 1e-6 (default)", cmfd=TUNED, fission_source_tol=0.0),
    solve("defaults", cmfd=TUNED),
]
report(balance_results, K_REFERENCE, PIN_REFERENCE)

With a loose balance tolerance and no fission-source test, the solve stops when the eigenvalue
change first falls below `k_tol`. The eigenvalue is already accurate, but the pin fission rates
are not. The default balance tolerance needs a few more power iterations and gives pin fission
rates that are more than an order of magnitude more accurate. The defaults, with both checks,
give the same accuracy.

## The coarse eigenvalue solve

In every power iteration, CMFD solves its coarse k-eigenvalue problem with its own power
iteration. It stops when the relative change of the coarse eigenvalue is below `pi_k_tol` and the
estimated error of the coarse flux shape is below `pi_phi_tol`, or after `pi_max_its` iterations.
Each coarse iteration is one coarse linear solve, a back substitution with the direct solver.
The eigenvalue converges much faster than the shape, and an unconverged coarse shape leaves the
slow modes unaccelerated. Limiting the coarse solve to one iteration shows this:

In [ ]:
coarse_results = [
    {**balance_results[2], "label": "coarse PI default"},
    solve("coarse PI 1 iteration", cmfd={**TUNED, "pi_max_its": 1}),
]
report(coarse_results, K_REFERENCE, PIN_REFERENCE)

## The coarse linear solver

`coarse_solver_policy="auto"` (the default) uses a direct PETSc LU factorization for coarse
problems below `coarse_direct_solve_threshold` global unknowns (150000 by default) and GMRES with
hypre BoomerAMG above it. `"direct"` and `"iterative"` force one path, and `"petsc_options"`
takes the solver from `petsc_options`. The coarse problem here has 664 coarse cells and two
groups, so `auto` chooses LU.

In [ ]:
solver_results = [
    {**balance_results[2], "label": "coarse LU (auto)"},
    solve("coarse GMRES + AMG", cmfd={**TUNED, "coarse_solver_policy": "iterative"}),
]
report(solver_results, K_REFERENCE, PIN_REFERENCE)

Both solvers give the same power iterations. LU is faster for a coarse problem of this size.
The iterative solver matters when the coarse problem is too large to factor. A coarse solve,
direct or iterative, counts as converged only if its true residual is also small. If a coarse
solve does not converge, CMFD skips that correction (`coarse_linear_solve_not_converged`). If the
coarse solve reached its iteration limit, raise `max_iters` in the `cmfd` options. That option
limits the coarse linear iterations. The `max_iters` argument of `solve()` limits the power
iterations instead. For other coarse-solve failures, use `coarse_solver_policy="direct"`, a stronger
`petsc_options` setting, or larger aggregates.

## Diagnostics

`verbose=True` on `CMFDAcceleration` prints stable `CMFD_ACCEL` lines for regression tests and
performance studies: the coarse-mesh statistics at initialization, and for early and periodic
later iterations, the correction (relaxation, damping, attempts, eigenvalues) and the time of each
CMFD phase (restriction, homogenization, assembly, coarse solve, prolongation and transport). The
next solve stops after three power iterations to keep the output short.

In [ ]:
verbose_result = solve("verbose", cmfd={**TUNED, "verbose": True}, max_iters=3)

After a solve, these accessors summarize it:

* `PowerIterationKEigenSolver.IsConverged()`: whether both the eigenvalue and the balance check
  converged,
* `GetNumPowerIterations()` and `GetNumSweeps()`: the work done,
* `CMFDAcceleration.GetNumCoarseCells()`: the size of the coarse mesh,
* `GetNumAppliedCorrections()` and `GetNumSkippedCorrections()`: how many corrections were applied
  and skipped.

## Skipped corrections

A skipped correction leaves the unaccelerated transport update for that power iteration. The
`skipped` column of the table below counts skipped corrections. The reason appears in the log when
the problem prints outer iterations, which `solve(..., verbose=True)` turns on, for example

```text
CMFD convergence check = not_satisfied, ..., correction = skipped (sign_change_guard)
```

`sign_change_guard` means every damped attempt would have changed the sign of the coarse flux in
some coarse cell and group. It is most common in the first power iteration, when the flux is still
close to the initial guess. Here, with all seven groups in the coarse problem and a five-iteration
GMRES update, the first correction is rejected:

In [ ]:
skip_results = [
    solve(
        "GMRES 5 update",
        cmfd={**TUNED, "group_aggregation_size": 1, "update_wgs_max_its": 5},
        inner_linear_method="petsc_gmres",
    ),
    solve(
        "GMRES 5 update, 1 inactive",
        cmfd={
            **TUNED,
            "group_aggregation_size": 1,
            "update_wgs_max_its": 5,
            "inactive_iterations": 1,
        },
        inner_linear_method="petsc_gmres",
    ),
]
report(skip_results, K_REFERENCE, PIN_REFERENCE)

An occasional early skip is harmless. Persistent skips mean that CMFD is not accelerating the
solve. Reduce `relaxation`, use smaller aggregates, or check the coarse solver.
`inactive_iterations` delays the first correction by a number of power iterations, which here
avoids the early skip. Inactive iterations use the groupset's own `l_max_its` and `l_abs_tol`. It is a debugging aid. Do not use it to hide repeated skips.

## Adjoint problems

CMFD also accelerates the adjoint k-eigenvalue problem. It homogenizes the transposed scattering
and production, and the adjoint eigenvalue equals the forward one.

In [ ]:
adjoint = solve("adjoint", cmfd=TUNED, adjoint=True)
results = (
    [reference] + balance_results + coarse_results + solver_results + skip_results + [adjoint]
)
report([adjoint], K_REFERENCE)

In [ ]:
by_label = {r["label"]: r for r in results}
for label, result in by_label.items():
    assert result["converged"], label
    assert abs(result["k"] - K_REFERENCE) < 1.0e-6, label
loose, default, defaults = balance_results
assert max_pin_error(loose, PIN_REFERENCE) > 10 * max_pin_error(default, PIN_REFERENCE)
assert max_pin_error(loose, PIN_REFERENCE) > 10 * max_pin_error(defaults, PIN_REFERENCE)
assert by_label["coarse PI 1 iteration"]["outers"] > by_label["coarse PI default"]["outers"]
assert by_label["GMRES 5 update"]["skipped"] >= 1
assert by_label["GMRES 5 update, 1 inactive"]["skipped"] == 0
if rank == 0:
    print(f"Loose balance max pin error={max_pin_error(loose, PIN_REFERENCE):.3e}")
    print(f"Default balance max pin error={max_pin_error(default, PIN_REFERENCE):.3e}")

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()